In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import os
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np


In [2]:
!pip install kaggle  # installs the Kaggle library

# Create folder
!mkdir -p mammals_dataset  # creates a folder to store the mammal dataset

# Download dataset
!kaggle datasets download -d asaniczka/mammals-image-classification-dataset-45-animals -p mammals_dataset  # downloads the mammal dataset

# Unzip dataset
!unzip mammals_dataset/mammals-image-classification-dataset-45-animals.zip -d mammals_dataset  # extracts the dataset

Streaming output truncated to the last 5000 lines.
  inflating: mammals_dataset/mammals/red_panda/red_panda-0250.jpg  
  inflating: mammals_dataset/mammals/red_panda/red_panda-0251.jpg  
  inflating: mammals_dataset/mammals/red_panda/red_panda-0252.jpg  
  inflating: mammals_dataset/mammals/red_panda/red_panda-0253.jpg  
  inflating: mammals_dataset/mammals/red_panda/red_panda-0254.jpg  
  inflating: mammals_dataset/mammals/red_panda/red_panda-0255.jpg  
  inflating: mammals_dataset/mammals/red_panda/red_panda-0256.jpg  
  inflating: mammals_dataset/mammals/red_panda/red_panda-0257.jpg  
  inflating: mammals_dataset/mammals/red_panda/red_panda-0258.jpg  
  inflating: mammals_dataset/mammals/red_panda/red_panda-0259.jpg  
  inflating: mammals_dataset/mammals/red_panda/red_panda-0260.jpg  
  inflating: mammals_dataset/mammals/red_panda/red_panda-0261.jpg  
  inflating: mammals_dataset/mammals/red_panda/red_panda-0262.jpg  
  inflating: mammals_dataset/mammals/red_panda/red_panda-0263.jpg

In [3]:
img_size = (224, 224)  # Standard size for most pretrained CNNs
batch_size = 32

In [6]:
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_data = datagen.flow_from_directory(

    '/content/mammals_dataset/mammals',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

Found 11020 images belonging to 45 classes.


In [7]:
val_data = datagen.flow_from_directory(
     '/content/mammals_dataset/mammals',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

Found 2731 images belonging to 45 classes.


In [21]:
base = MobileNetV2(
    include_top=False,  # removes the original classification layers
    weights='imagenet',  # uses pre-trained ImageNet weights
    input_shape=(224, 224, 3)  # sets input image size
)

base.trainable = False  # freezes all base layers initially

x = GlobalAveragePooling2D()(base.output)  # converts feature maps into a single vector

x = Dense(256, activation='relu')(x)  # adds a custom dense layer with 256 neurons

output = Dense(45, activation='softmax')(x)  # output layer with 45 neurons for 45 mammal classes

model = Model(
    inputs=base.input,
    outputs=output
)  # creates the complete model

model.compile(
    optimizer=Adam(0.0001),  # sets learning rate
    loss='categorical_crossentropy',  # calculates error for 45 classes
    metrics=['accuracy']  # measures prediction accuracy
)

model.summary()  # displays model information



Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer_2[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,597,485 (9.91 MB)

 Trainable params: 339,501 (1.30 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [23]:
base.trainable = True  # unfreezes the MobileNetV2 layers

for layer in base.layers[:-20]:  # selects all layers except the last 20
    layer.trainable = False  # keeps earlier layers frozen

for layer in base.layers[-20:]:  # selects the last 20 layers
    layer.trainable = True  # allows the last 20 layers to learn

for layer in base.layers:  # goes through all MobileNetV2 layers
    print(layer.name, layer.trainable)  # prints layer name and trainable status

input_layer_2 False
Conv1 False
bn_Conv1 False
Conv1_relu False
expanded_conv_depthwise False
expanded_conv_depthwise_BN False
expanded_conv_depthwise_relu False
expanded_conv_project False
expanded_conv_project_BN False
block_1_expand False
block_1_expand_BN False
block_1_expand_relu False
block_1_pad False
block_1_depthwise False
block_1_depthwise_BN False
block_1_depthwise_relu False
block_1_project False
block_1_project_BN False
block_2_expand False
block_2_expand_BN False
block_2_expand_relu False
block_2_depthwise False
block_2_depthwise_BN False
block_2_depthwise_relu False
block_2_project False
block_2_project_BN False
block_2_add False
block_3_expand False
block_3_expand_BN False
block_3_expand_relu False
block_3_pad False
block_3_depthwise False
block_3_depthwise_BN False
block_3_depthwise_relu False
block_3_project False
block_3_project_BN False
block_4_expand False
block_4_expand_BN False
block_4_expand_relu False
block_4_depthwise False
block_4_depthwise_BN False
block_4_d

In [24]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
345/345 ━━━━━━━━━━━━━━━━━━━━ 52s 112ms/step - accuracy: 0.7586 - loss: 1.0710 - val_accuracy: 0.8693 - val_loss: 0.4699
Epoch 2/10
345/345 ━━━━━━━━━━━━━━━━━━━━ 23s 67ms/step - accuracy: 0.9300 - loss: 0.2586 - val_accuracy: 0.8832 - val_loss: 0.4026
Epoch 3/10
345/345 ━━━━━━━━━━━━━━━━━━━━ 24s 68ms/step - accuracy: 0.9699 - loss: 0.1210 - val_accuracy: 0.9074 - val_loss: 0.3222
Epoch 4/10
345/345 ━━━━━━━━━━━━━━━━━━━━ 23s 67ms/step - accuracy: 0.9883 - loss: 0.0607 - val_accuracy: 0.9147 - val_loss: 0.3146
Epoch 5/10
345/345 ━━━━━━━━━━━━━━━━━━━━ 25s 72ms/step - accuracy: 0.9948 - loss: 0.0349 - val_accuracy: 0.9143 - val_loss: 0.3103
Epoch 6/10
345/345 ━━━━━━━━━━━━━━━━━━━━ 23s 67ms/step - accuracy: 0.9962 - loss: 0.0263 - val_accuracy: 0.9161 - val_loss: 0.3016
Epoch 7/10
345/345 ━━━━━━━━━━━━━━━━━━━━ 23s 68ms/step - accuracy: 0.9956 - loss: 0.0231 - val_accuracy: 0.9161 - val_loss: 0.3288
Epoch 8/10
345/345 ━━━━━━━━━━━━━━━━━━━━ 25s 72ms/step - accuracy: 0.9967 - loss: 0.0189 -

In [25]:
print(train_data.class_indices)

{'african_elephant': 0, 'alpaca': 1, 'american_bison': 2, 'anteater': 3, 'arctic_fox': 4, 'armadillo': 5, 'baboon': 6, 'badger': 7, 'blue_whale': 8, 'brown_bear': 9, 'camel': 10, 'dolphin': 11, 'giraffe': 12, 'groundhog': 13, 'highland_cattle': 14, 'horse': 15, 'jackal': 16, 'kangaroo': 17, 'koala': 18, 'manatee': 19, 'mongoose': 20, 'mountain_goat': 21, 'opossum': 22, 'orangutan': 23, 'otter': 24, 'polar_bear': 25, 'porcupine': 26, 'red_panda': 27, 'rhinoceros': 28, 'sea_lion': 29, 'seal': 30, 'snow_leopard': 31, 'squirrel': 32, 'sugar_glider': 33, 'tapir': 34, 'vampire_bat': 35, 'vicuna': 36, 'walrus': 37, 'warthog': 38, 'water_buffalo': 39, 'weasel': 40, 'wildebeest': 41, 'wombat': 42, 'yak': 43, 'zebra': 44}


In [26]:
model = load_model("mammals_model.h5")  # loads the trained model

img = image.load_img(
    "/content/images (3).jpg",
    target_size=(224, 224)
)  # loads and resizes the image

img = np.array(img) / 255.0  # normalizes the image

img = np.expand_dims(img, axis=0)  # adds batch dimension

prediction = model.predict(img)  # predicts the mammal

classes = list(train_data.class_indices.keys())  # gets mammal class names

print("Mammal:", classes[np.argmax(prediction)])  # prints the predicted mammal

1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
Mammal: yak
